# Notebook 23 — Full Project: Station Alignment with ArUco

---

**Part 8 · Robotics Projects**

```
┌──────────────────────────────────────────────────────────────────────┐
│   Full Autonomous Docking System                                     │
│                                                                      │
│   Camera ──► Detect ArUco ──► Compute Offsets ──► Generate Command  │
│              (6D pose)         (lateral, heading,   (move, rotate,   │
│                                 distance errors)     stop, status)   │
└──────────────────────────────────────────────────────────────────────┘
```

> **This is a full robotics project**: take everything from Parts 2–5 and assemble a complete, production-style station alignment system. The robot sees an ArUco marker, computes its pose, and generates navigation commands to dock precisely.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install numpy matplotlib opencv-contrib-python -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple
from enum import Enum
import time
print('All imports OK')

## Learning Objectives

By the end of this notebook you will:

1. Understand the complete **autonomous docking pipeline** end-to-end  
2. Implement a **multi-station registry** with per-station parameters  
3. Compute **3-axis alignment errors** (lateral, heading, distance) from ArUco pose  
4. Build a **proportional controller** that outputs robot commands  
5. Add **state machine logic** (SEARCHING → APPROACHING → ALIGNING → DOCKED)  
6. Create a full **HUD overlay** showing real-time alignment status  
7. Simulate the complete pipeline on synthetic camera data

## 1 — System Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    Autonomous Station Alignment System                      │
│                                                                             │
│  ┌──────────┐    ┌──────────────┐    ┌───────────────┐    ┌─────────────┐  │
│  │  Camera  │───►│ ArUco Detect │───►│ Pose Estimate │───►│ Alignment   │  │
│  │  (RGB)   │    │ & Track      │    │ estimatePose  │    │ Error Calc  │  │
│  └──────────┘    └──────────────┘    └───────────────┘    └──────┬──────┘  │
│                                                                   │         │
│  ┌──────────┐    ┌──────────────┐    ┌───────────────┐           │         │
│  │  Robot   │◄───│ Command Gen  │◄───│ State Machine │◄──────────┘         │
│  │  Motors  │    │ P-Controller │    │ SEARCH/DOCK   │                     │
│  └──────────┘    └──────────────┘    └───────────────┘                     │
│                          │                                                  │
│                          ▼                                                  │
│                    ┌──────────┐                                             │
│                    │  HUD     │ (display on camera feed)                    │
│                    └──────────┘                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Data Flow

```
Camera frame
  → [ArUco detector] → corners, ids, rvecs, tvecs
  → [Pose interpreter] → lateral_error, heading_error, distance
  → [State machine] → current_state
  → [P-controller] → forward_speed, lateral_speed, turn_rate
  → [HUD renderer] → annotated frame for display
```

In [ ]:
# ── ArUco API compatibility shim ──────────────────────────────────────────

def get_aruco_dict(dict_id=cv2.aruco.DICT_4X4_50):
    try:
        return cv2.aruco.getPredefinedDictionary(dict_id)   # OpenCV ≥ 4.7
    except AttributeError:
        return cv2.aruco.Dictionary_get(dict_id)            # OpenCV ≤ 4.6

def get_aruco_params():
    try:
        return cv2.aruco.DetectorParameters()
    except AttributeError:
        return cv2.aruco.DetectorParameters_create()

def detect_markers(gray, aruco_dict, params):
    try:
        detector = cv2.aruco.ArucoDetector(aruco_dict, params)
        corners, ids, rejected = detector.detectMarkers(gray)
    except AttributeError:
        corners, ids, rejected = cv2.aruco.detectMarkers(gray, aruco_dict, parameters=params)
    return corners, ids, rejected

ARUCO_DICT = get_aruco_dict()
ARUCO_PARAMS = get_aruco_params()
print('ArUco ready (API compatibility shim loaded)')

## 2 — Station Registry

Each docking station has:
- A unique **ArUco marker ID**
- A **marker size** (for accurate pose estimation)
- A **dock distance** — how close to approach before stopping
- A **name** and **type** for logging
- Optional **approach angle** offset (if you need to approach from a specific direction)

In [ ]:
# ── Station Registry ──────────────────────────────────────────────────────

@dataclass
class Station:
    """Represents a docking station with an ArUco marker."""
    station_id:     int
    aruco_id:       int
    name:           str
    marker_size_m:  float   # physical marker size in meters
    dock_dist_m:    float   # final docking distance in meters
    approach_dist_m: float  # distance to start precision alignment
    approach_angle_deg: float = 0.0  # required approach angle offset (degrees)
    priority:       int = 1  # 1=normal, 0=charging (higher priority)

class DockState(Enum):
    SEARCHING   = 'SEARCHING'
    APPROACHING = 'APPROACHING'
    ALIGNING    = 'ALIGNING'
    DOCKED      = 'DOCKED'
    LOST        = 'LOST'

# ── Define all stations ───────────────────────────────────────────────────
STATION_REGISTRY: Dict[int, Station] = {
    0: Station(station_id=0, aruco_id=0,
               name='Charging Station',
               marker_size_m=0.15, dock_dist_m=0.30,
               approach_dist_m=1.0, priority=0),

    1: Station(station_id=1, aruco_id=1,
               name='Conveyor A',
               marker_size_m=0.15, dock_dist_m=0.25,
               approach_dist_m=0.8, priority=1),

    2: Station(station_id=2, aruco_id=2,
               name='Conveyor B',
               marker_size_m=0.15, dock_dist_m=0.25,
               approach_dist_m=0.8, priority=1),

    3: Station(station_id=3, aruco_id=3,
               name='Inspection Bay',
               marker_size_m=0.20, dock_dist_m=0.50,
               approach_dist_m=1.5,
               approach_angle_deg=90.0,  # approach from the side
               priority=1),

    4: Station(station_id=4, aruco_id=4,
               name='Parts Dispenser',
               marker_size_m=0.10, dock_dist_m=0.20,
               approach_dist_m=0.6, priority=1),
}

print('Station Registry:')
print(f'{"ID":>4}  {"ArUco":>6}  {"Name":<20}  {"Marker":>8}  {"Dock Dist":>10}  {"Priority"}')
print('-' * 68)
for sid, s in STATION_REGISTRY.items():
    print(f'{sid:>4}  {s.aruco_id:>6}  {s.name:<20}  {s.marker_size_m*100:>6.0f}cm  '
          f'{s.dock_dist_m*100:>8.0f}cm  {"PRIORITY" if s.priority==0 else "normal"}')

## 3 — Alignment Error Computation

From `estimatePoseSingleMarkers` we get `tvec` and `rvec`:

```
tvec = [X, Y, Z] in camera frame:
  X > 0 → marker is to the RIGHT
  X < 0 → marker is to the LEFT
  Y > 0 → marker is BELOW center
  Z     → depth (distance to marker)

rvec → rotation matrix → extract heading angle:
  heading_error = angle between robot's Z and marker's normal
```

For docking, we want:
- `X ≈ 0` → centered laterally
- `heading_error ≈ 0` → facing the station straight-on
- `Z ≈ dock_dist_m` → at the right distance

In [ ]:
# ── Alignment Error Computation ───────────────────────────────────────────

@dataclass
class AlignmentError:
    lateral_m:   float   # X offset: + = robot must move RIGHT, - = LEFT
    distance_m:  float   # Z: current distance to marker
    dist_error_m: float  # distance - dock_dist (+ = too far, - = overshot)
    heading_deg: float   # heading error in degrees
    is_aligned:  bool    # within all tolerances

@dataclass
class RobotCommand:
    forward:  float   # m/s, + = forward
    lateral:  float   # m/s, + = right
    turn:     float   # rad/s, + = turn right
    state:    DockState
    message:  str

# Controller gains
KP_DIST   = 0.5   # proportional gain for distance control
KP_LAT    = 1.2   # proportional gain for lateral control
KP_HEAD   = 0.05  # proportional gain for heading (deg → rad/s)

# Tolerances (consider "aligned" when all errors below these)
TOL_LATERAL_M  = 0.02   # 2 cm lateral tolerance
TOL_DIST_M     = 0.03   # 3 cm distance tolerance
TOL_HEADING_DEG = 3.0   # 3 degree heading tolerance

def compute_heading_error(rvec: np.ndarray) -> float:
    """
    Extract the heading error (rotation around Y axis) from rvec.
    Returns degrees. Positive = marker tilted clockwise from robot's view.
    """
    R, _ = cv2.Rodrigues(rvec)
    # The marker's Z-axis (normal) in camera frame is R's 3rd column
    marker_normal = R[:, 2]  # [nx, ny, nz]
    # Heading error = angle between camera's Z axis and marker normal, projected to XZ plane
    heading_rad = np.arctan2(marker_normal[0], marker_normal[2])
    return np.degrees(heading_rad)

def compute_alignment_error(tvec: np.ndarray, rvec: np.ndarray,
                             station: Station) -> AlignmentError:
    """Compute 3-axis alignment error from pose to station dock point."""
    lateral_m    = float(tvec[0])        # X offset
    distance_m   = float(tvec[2])        # Z (depth)
    dist_error_m = distance_m - station.dock_dist_m
    heading_deg  = compute_heading_error(rvec)

    is_aligned = (
        abs(lateral_m)    < TOL_LATERAL_M  and
        abs(dist_error_m) < TOL_DIST_M     and
        abs(heading_deg)  < TOL_HEADING_DEG
    )

    return AlignmentError(
        lateral_m=lateral_m,
        distance_m=distance_m,
        dist_error_m=dist_error_m,
        heading_deg=heading_deg,
        is_aligned=is_aligned
    )

def generate_robot_command(error: AlignmentError) -> RobotCommand:
    """Proportional controller: maps alignment errors to robot commands."""
    # Clamp outputs to safe limits
    MAX_FWD   = 0.3   # m/s
    MAX_LAT   = 0.2   # m/s
    MAX_TURN  = 0.5   # rad/s

    if error.is_aligned:
        return RobotCommand(0.0, 0.0, 0.0, DockState.DOCKED, 'DOCKED')

    forward = np.clip(KP_DIST * error.dist_error_m,  -MAX_FWD,  MAX_FWD)
    lateral = np.clip(KP_LAT * error.lateral_m,      -MAX_LAT,  MAX_LAT)
    turn    = np.clip(KP_HEAD * error.heading_deg,   -MAX_TURN, MAX_TURN)

    # Determine state
    if abs(error.dist_error_m) > 0.3:
        state = DockState.APPROACHING
        msg = f'APPROACHING  Z={error.distance_m:.2f}m'
    else:
        state = DockState.ALIGNING
        msg = f'ALIGNING  lat={error.lateral_m*100:.1f}cm  hdg={error.heading_deg:.1f}°'

    return RobotCommand(forward=forward, lateral=lateral, turn=turn,
                        state=state, message=msg)

# ── Test with sample pose data ─────────────────────────────────────────────
station = STATION_REGISTRY[1]   # Conveyor A, dock_dist=0.25m

test_cases = [
    (np.array([0.08, 0.0, 0.80]), np.array([0.1, -0.05, 0.0]), 'Approaching, slightly right'),
    (np.array([0.01, 0.0, 0.30]), np.array([0.05, 0.0, 0.0]),  'Aligning, nearly there'),
    (np.array([0.005, 0.0, 0.25]), np.array([0.02, 0.0, 0.0]), 'Docked!'),
]

print('═' * 70)
print('  Alignment Error + Command Generator Test')
print('═' * 70)
for tvec, rvec, description in test_cases:
    err = compute_alignment_error(tvec, rvec, station)
    cmd = generate_robot_command(err)
    print(f'\n  [{description}]')
    print(f'    Lateral: {err.lateral_m*100:+.1f} cm  Dist error: {err.dist_error_m*100:+.1f} cm  '
          f'Heading: {err.heading_deg:+.1f}°  Aligned: {err.is_aligned}')
    print(f'    Cmd → fwd={cmd.forward:+.3f} lat={cmd.lateral:+.3f} turn={cmd.turn:+.3f}  '
          f'State: {cmd.state.value}')
print('═' * 70)

## 4 — State Machine

```
                 ┌────────────┐
    start ──────►│ SEARCHING  │◄─── marker lost for > N frames
                 └─────┬──────┘
                       │ marker found
                       ▼
                 ┌────────────┐
                 │ APPROACHING│ (dist_error > 30cm)
                 └─────┬──────┘
                       │ dist_error < 30cm
                       ▼
                 ┌────────────┐
                 │ ALIGNING   │ (fine-tune lateral + heading)
                 └─────┬──────┘
                       │ all errors within tolerance
                       ▼
                 ┌────────────┐
                 │  DOCKED    │ (stop all motors)
                 └────────────┘
```

In [ ]:
# ── State Machine ─────────────────────────────────────────────────────────

class DockingStateMachine:
    def __init__(self, target_station_id: int, lost_timeout_frames: int = 10):
        self.target = STATION_REGISTRY[target_station_id]
        self.state = DockState.SEARCHING
        self.frames_since_detection = 0
        self.lost_timeout = lost_timeout_frames
        self.docked_at = None

    def update(self, tvec=None, rvec=None) -> RobotCommand:
        """
        Update state machine with latest pose data.
        tvec/rvec = None if marker not detected this frame.
        """
        if tvec is None or rvec is None:
            self.frames_since_detection += 1
            if self.frames_since_detection >= self.lost_timeout:
                if self.state != DockState.DOCKED:
                    self.state = DockState.LOST
            return RobotCommand(0.0, 0.0, 0.0, self.state,
                                f'No marker ({self.frames_since_detection} frames)')

        self.frames_since_detection = 0

        error = compute_alignment_error(tvec, rvec, self.target)
        cmd = generate_robot_command(error)
        self.state = cmd.state

        return cmd

# ── Simulate a full docking sequence ──────────────────────────────────────

def simulate_docking_sequence(target_station_id=1, n_frames=40):
    """
    Simulate the robot approaching and docking at a station.
    Returns list of (frame_idx, tvec, cmd) tuples.
    """
    sm = DockingStateMachine(target_station_id)
    station = STATION_REGISTRY[target_station_id]

    # Simulate robot starting 1.2m away, 10cm left, 5° off heading
    z0, z_target = 1.2, station.dock_dist_m
    x0, x_target = -0.10, 0.0
    h0, h_target = 8.0, 0.0

    history = []
    for i in range(n_frames):
        alpha = i / (n_frames - 1)   # 0 → 1 over the sequence
        # Simulate pose converging (as P-controller drives errors to zero)
        z = z0 + (z_target - z0) * min(alpha * 1.5, 1.0)
        x = x0 * (1 - min(alpha * 2.0, 1.0))
        h = h0 * (1 - min(alpha * 2.5, 1.0))
        tvec = np.array([x, 0.0, z])
        rvec = np.array([0.0, np.radians(h), 0.0])

        # Drop detection for frames 8-10 (simulate brief occlusion)
        if 8 <= i <= 9:
            cmd = sm.update(None, None)
        else:
            cmd = sm.update(tvec, rvec)

        history.append({'frame': i, 'tvec': tvec.copy(), 'cmd': cmd})

    return history

history = simulate_docking_sequence(target_station_id=1, n_frames=40)

# Plot state timeline and errors
frames = [h['frame'] for h in history]
z_vals = [h['tvec'][2] for h in history]
x_vals = [h['tvec'][0] * 100 for h in history]   # cm
fwd_vals = [h['cmd'].forward for h in history]
states = [h['cmd'].state.value for h in history]
state_colors = {'SEARCHING': '#95a5a6', 'APPROACHING': '#e74c3c',
                'ALIGNING': '#f39c12', 'DOCKED': '#2ecc71', 'LOST': '#8e44ad'}

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

ax1 = axes[0]
ax1.plot(frames, z_vals, 'b-o', ms=4, lw=2, label='Z distance (m)')
ax1.axhline(STATION_REGISTRY[1].dock_dist_m, color='green', ls='--', lw=1.5, label='Dock target')
ax1.axhspan(STATION_REGISTRY[1].dock_dist_m - TOL_DIST_M,
             STATION_REGISTRY[1].dock_dist_m + TOL_DIST_M,
             alpha=0.15, color='green', label='Tolerance')
ax1.set_ylabel('Distance Z (m)')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_title('Docking Simulation: Station 1 (Conveyor A)', fontweight='bold')

ax2 = axes[1]
ax2.plot(frames, x_vals, 'r-o', ms=4, lw=2, label='Lateral X (cm)')
ax2.axhline(0, color='black', ls='--', lw=1)
ax2.axhspan(-TOL_LATERAL_M*100, TOL_LATERAL_M*100, alpha=0.15, color='green')
ax2.set_ylabel('Lateral offset (cm)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

ax3 = axes[2]
for h in history:
    ax3.bar(h['frame'], 1, color=state_colors.get(h['cmd'].state.value, 'gray'),
            edgecolor='none', linewidth=0)
# Legend
patches = [mpatches.Patch(color=c, label=s) for s, c in state_colors.items()
           if s in set(states)]
ax3.legend(handles=patches, fontsize=8, loc='upper right')
ax3.set_ylabel('State')
ax3.set_xlabel('Frame')
ax3.set_yticks([])

plt.tight_layout()
plt.show()

print(f'\nFinal state: {history[-1]["cmd"].state.value}')
print(f'Final message: {history[-1]["cmd"].message}')

## 5 — HUD Overlay

The HUD (Heads-Up Display) renders all docking information on the camera feed.

In [ ]:
# ── HUD Renderer ──────────────────────────────────────────────────────────

STATE_COLORS_BGR = {
    DockState.SEARCHING:   (128, 128, 128),
    DockState.APPROACHING: (0,   80,  255),
    DockState.ALIGNING:    (0,   165, 255),
    DockState.DOCKED:      (0,   200,  50),
    DockState.LOST:        (0,     0, 200),
}

def draw_docking_hud(frame: np.ndarray, cmd: RobotCommand,
                     error: Optional[AlignmentError],
                     station: Station) -> np.ndarray:
    """Render full docking HUD onto a copy of frame."""
    out = frame.copy()
    h, w = out.shape[:2]
    state_color = STATE_COLORS_BGR[cmd.state]

    # Header bar
    cv2.rectangle(out, (0, 0), (w, 50), (20, 20, 30), -1)
    cv2.putText(out, f'TARGET: {station.name}  |  STATE: {cmd.state.value}',
                (10, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.7, state_color, 2)

    if error is not None:
        # Lateral alignment bar (centered bar at bottom)
        bar_y, bar_h = h - 60, 20
        bar_w = 300
        bar_x = w // 2 - bar_w // 2
        cv2.rectangle(out, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (50, 50, 60), -1)
        cv2.rectangle(out, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (120, 120, 140), 1)
        # Center tick
        cv2.line(out, (w//2, bar_y), (w//2, bar_y + bar_h), (200, 200, 200), 1)
        # Lateral offset indicator (clamp to bar)
        lat_px = int(np.clip(error.lateral_m / 0.3 * (bar_w // 2), -bar_w//2, bar_w//2))
        ind_x = w // 2 + lat_px
        ind_color = (0, 200, 50) if abs(error.lateral_m) < TOL_LATERAL_M else (0, 100, 255)
        cv2.circle(out, (ind_x, bar_y + bar_h // 2), 8, ind_color, -1)
        cv2.putText(out, f'Lateral: {error.lateral_m*100:+.1f} cm',
                    (bar_x, bar_y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

        # Distance text
        dist_color = (0, 200, 50) if abs(error.dist_error_m) < TOL_DIST_M else (0, 100, 255)
        cv2.putText(out, f'Distance: {error.distance_m:.3f}m  (target {station.dock_dist_m:.2f}m)',
                    (10, h - 80), cv2.FONT_HERSHEY_SIMPLEX, 0.55, dist_color, 1)

        # Heading
        head_color = (0, 200, 50) if abs(error.heading_deg) < TOL_HEADING_DEG else (0, 100, 255)
        cv2.putText(out, f'Heading: {error.heading_deg:+.1f} deg',
                    (10, h - 100), cv2.FONT_HERSHEY_SIMPLEX, 0.55, head_color, 1)

    # Command display
    cv2.putText(out, f'CMD  fwd:{cmd.forward:+.2f}  lat:{cmd.lateral:+.2f}  turn:{cmd.turn:+.2f}',
                (10, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

    # DOCKED banner
    if cmd.state == DockState.DOCKED:
        cv2.putText(out, 'DOCKED', (w//2 - 90, h//2),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 255, 80), 4)

    return out

# ── Render sample HUD frames ───────────────────────────────────────────────
K = np.array([[600, 0, 320], [0, 600, 240], [0, 0, 1]], dtype=np.float64)
dist_coeffs = np.zeros(4)
station = STATION_REGISTRY[1]

sample_frames = [
    {'tvec': np.array([ 0.10, 0.0, 0.90]), 'rvec': np.array([0.0, 0.14, 0.0]), 'label': 'Approaching'},
    {'tvec': np.array([-0.03, 0.0, 0.35]), 'rvec': np.array([0.0, 0.05, 0.0]), 'label': 'Aligning'},
    {'tvec': np.array([ 0.005, 0.0, 0.25]), 'rvec': np.array([0.0, 0.01, 0.0]), 'label': 'Docked'},
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sm = DockingStateMachine(target_station_id=1)

for ax, sf in zip(axes, sample_frames):
    # Create blank frame with ArUco-like overlay
    frame = np.full((480, 640, 3), (40, 40, 50), dtype=np.uint8)
    cv2.rectangle(frame, (250, 180), (390, 300), (90, 110, 130), -1)
    cv2.rectangle(frame, (250, 180), (390, 300), (150, 180, 200), 2)
    cv2.putText(frame, 'ArUco ID=1', (255, 320),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 200, 220), 1)

    # Draw axes
    try:
        rvec = sf['rvec'].astype(np.float32)
        tvec = sf['tvec'].astype(np.float32)
        cv2.drawFrameAxes(frame, K, dist_coeffs, rvec, tvec, 0.07)
    except Exception:
        pass

    # Compute error and command
    err = compute_alignment_error(sf['tvec'], sf['rvec'], station)
    cmd = generate_robot_command(err)

    # Render HUD
    frame_hud = draw_docking_hud(frame, cmd, err, station)

    ax.imshow(cv2.cvtColor(frame_hud, cv2.COLOR_BGR2RGB))
    ax.set_title(sf['label'], fontweight='bold')
    ax.axis('off')

plt.suptitle('Station Alignment HUD — Three Docking Stages', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6 — Production Deployment Notes

### Connecting to a Real Camera

```python
cap = cv2.VideoCapture(0)    # USB camera
# For RealSense: use pyrealsense2
# For ZED: use pyzed

sm = DockingStateMachine(target_station_id=1)
K = load_calibration('camera_calibration.npz')

while True:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    corners, ids, _ = detect_markers(gray, ARUCO_DICT, ARUCO_PARAMS)

    if ids is not None:
        target_id = sm.target.aruco_id
        if target_id in ids.flatten():
            idx = list(ids.flatten()).index(target_id)
            rvecs, tvecs, _ = cv2.aruco.estimatePoseSingleMarkers(
                corners[idx:idx+1], sm.target.marker_size_m, K, np.zeros(4))
            cmd = sm.update(tvecs[0][0], rvecs[0][0])
        else:
            cmd = sm.update(None, None)
    else:
        cmd = sm.update(None, None)

    send_to_robot(cmd)   # your robot's API
    cv2.imshow('Docking', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
```

### Calibration Reminder

For accurate pose, always use a **calibrated camera**. Load from the calibration file we created in NB 07:

```python
data = np.load('../assets/calibration/camera_calibration.npz')
K    = data['K']
dist = data['dist']
```

## Recap

| Component | Key Point |
|-----------|----------|
| Station Registry | Dict of `Station` dataclasses with per-station parameters |
| Alignment errors | lateral (X), dist_error (Z-dock_dist), heading (rvec → Y rotation) |
| P-controller | `cmd = K_p × error`, clamped to safe velocity limits |
| State machine | SEARCHING → APPROACHING → ALIGNING → DOCKED |
| Lost handling | Count missed frames; transition to LOST after timeout |
| HUD | Lateral bar, distance text, heading, state color, DOCKED banner |
| Production | Replace synthetic frames with `cv2.VideoCapture` + calibrated K |

In [ ]:
# ============================================================
# EXERCISE 1: Add a 4th station to the registry
# ============================================================
# Add "Assembly Bay" (ID=5, ArUco=5, 20cm marker, dock at 40cm,
# approach from 1.2m, approach angle 0°, priority 1).

# YOUR CODE HERE
# STATION_REGISTRY[5] = Station(...)

# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# STATION_REGISTRY[5] = Station(
#     station_id=5, aruco_id=5,
#     name='Assembly Bay',
#     marker_size_m=0.20, dock_dist_m=0.40,
#     approach_dist_m=1.2, priority=1
# )
# print(STATION_REGISTRY[5])

In [ ]:
# ============================================================
# EXERCISE 2: Tighten the alignment tolerances
# ============================================================
# The charging station (ID=0) requires extra precision.
# Change the tolerances to:
#   lateral = 0.5 cm, distance = 1 cm, heading = 1 deg
# Then re-run the 'Docked' test case to verify it still reaches DOCKED state.

# YOUR CODE HERE

# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# TOL_LATERAL_M  = 0.005
# TOL_DIST_M     = 0.01
# TOL_HEADING_DEG = 1.0
#
# station = STATION_REGISTRY[0]
# tvec = np.array([0.003, 0.0, 0.30])  # within tighter tolerances
# rvec = np.array([0.0, 0.01, 0.0])
# err = compute_alignment_error(tvec, rvec, station)
# print(f'Aligned: {err.is_aligned}')  # should be True